# 📖 Notebook 3: Choosing the Right Database for Your Use Case

Now that you understand the CAP trade-off and have seen it in action, let's talk about **choosing the right database** — the decision that comes up in every system design interview.

## Learning Objectives

By the end of this notebook, you'll understand:
- Which popular databases are CP vs AP
- The spectrum of consistency levels (not just strong vs eventual)
- How real systems mix CP and AP within the same application
- A decision framework for system design interviews

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/cap-theorem
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
# Make sure the replica is RUNNING (in case Notebook 2 left it paused).
import subprocess as _sp
_sp.run(["docker", "unpause", "cap-postgres-replica"], capture_output=True)

import psycopg2
import redis
import time
import json

PRIMARY_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "cap_demo",
    "user": "demo",
    "password": "demo",
    "connect_timeout": 3,
}

REPLICA_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "cap_demo",
    "user": "demo",
    "password": "demo",
    "connect_timeout": 3,
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_primary():
    return psycopg2.connect(**PRIMARY_CONFIG)

def get_replica():
    return psycopg2.connect(**REPLICA_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Verify connections
for name, fn in [("Primary", get_primary), ("Replica", get_replica)]:
    try:
        c = fn(); c.close()
        print(f"✅ {name} connected")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

✅ Primary connected
✅ Replica connected
✅ Redis connected


## 🗄️ The Database Landscape: CP vs AP

Every distributed database makes a choice on the CAP spectrum. Here's how popular databases line up:

```
           Consistency ◄──────────────────────────────────────────► Availability
              (CP)                                                    (AP)

    ┌──────────┐  ┌──────────┐  ┌──────────────┐  ┌──────────┐  ┌──────────┐
    │ Spanner  │  │ Postgres │  │  DynamoDB     │  │ Cassandra│  │  Redis   │
    │ (Google) │  │ (single  │  │  (configurable│  │          │  │ Cluster  │
    │          │  │  primary)│  │   per-query)  │  │          │  │          │
    └──────────┘  └──────────┘  └──────────────┘  └──────────┘  └──────────┘
       Strong        Strong       Strong OR           Eventual      Eventual
    consistency   consistency     Eventual           consistency   consistency
```

### Key Insight: Some Databases Let You Choose!

Modern databases like **DynamoDB** let you pick consistency **per query**:
- `ConsistentRead=True` → strong consistency (CP behavior)
- `ConsistentRead=False` → eventual consistency (AP behavior, faster)

This means the CP vs AP choice isn't always a one-time architectural decision — it can be made per feature.

In [2]:
# Let's build a reference table of popular databases and their CAP properties

databases = [
    ("PostgreSQL",     "CP",         "Strong",    "RDBMS with ACID transactions. Single primary, read replicas.",
     "Banking, e-commerce, any CRUD app"),
    ("MySQL",          "CP",         "Strong",    "Similar to Postgres. Widely used RDBMS.",
     "WordPress, web apps, SaaS platforms"),
    ("Google Spanner", "CP",         "Strong",    "Globally distributed SQL. Uses TrueTime (atomic clocks).",
     "Global financial systems, ad platforms"),
    ("MongoDB",        "CP",         "Strong*",   "Document store. Strong consistency with replica sets. *Configurable.",
     "Content management, catalogs, user data"),
    ("DynamoDB",       "Configurable", "Both",    "Key-value/document. Choose per-query: strong or eventual.",
     "Gaming leaderboards, IoT, session stores"),
    ("Cassandra",      "AP",         "Eventual",  "Wide-column store. Tunable consistency with quorum reads.",
     "Time-series data, messaging, IoT"),
    ("Redis",          "AP",         "Eventual",  "In-memory key-value. Blazing fast. Async replication in clusters.",
     "Caching, session store, real-time leaderboards"),
    ("CockroachDB",    "CP",         "Strong",    "Distributed SQL. Inspired by Spanner but open source.",
     "Multi-region apps needing SQL + consistency"),
]

print("🗄️  Database CAP Reference Guide")
print("=" * 90)
print()
print(f"{'Database':<16} {'CAP':<14} {'Consistency':<12} Description")
print("-" * 90)
for db, cap, consistency, desc, use_case in databases:
    icon = "🔒" if cap == "CP" else "🌐" if cap == "AP" else "⚙️"
    print(f"{icon} {db:<14} {cap:<14} {consistency:<12} {desc}")

print()
print("💡 In interviews, PostgreSQL (CP) and Cassandra/Redis (AP) are the most common choices.")
print("   DynamoDB is great to mention when you want per-query flexibility.")

🗄️  Database CAP Reference Guide

Database         CAP            Consistency  Description
------------------------------------------------------------------------------------------
🔒 PostgreSQL     CP             Strong       RDBMS with ACID transactions. Single primary, read replicas.
🔒 MySQL          CP             Strong       Similar to Postgres. Widely used RDBMS.
🔒 Google Spanner CP             Strong       Globally distributed SQL. Uses TrueTime (atomic clocks).
🔒 MongoDB        CP             Strong*      Document store. Strong consistency with replica sets. *Configurable.
⚙️ DynamoDB       Configurable   Both         Key-value/document. Choose per-query: strong or eventual.
🌐 Cassandra      AP             Eventual     Wide-column store. Tunable consistency with quorum reads.
🌐 Redis          AP             Eventual     In-memory key-value. Blazing fast. Async replication in clusters.
🔒 CockroachDB    CP             Strong       Distributed SQL. Inspired by Spanner but open so

In [3]:
# Print the use cases separately for clarity

print("📋 When to Use Each Database")
print("=" * 75)
print()
for db, cap, consistency, desc, use_case in databases:
    icon = "🔒" if cap == "CP" else "🌐" if cap == "AP" else "⚙️"
    print(f"{icon} {db}")
    print(f"   Use for: {use_case}")
    print()

📋 When to Use Each Database

🔒 PostgreSQL
   Use for: Banking, e-commerce, any CRUD app

🔒 MySQL
   Use for: WordPress, web apps, SaaS platforms

🔒 Google Spanner
   Use for: Global financial systems, ad platforms

🔒 MongoDB
   Use for: Content management, catalogs, user data

⚙️ DynamoDB
   Use for: Gaming leaderboards, IoT, session stores

🌐 Cassandra
   Use for: Time-series data, messaging, IoT

🌐 Redis
   Use for: Caching, session store, real-time leaderboards

🔒 CockroachDB
   Use for: Multi-region apps needing SQL + consistency



## 🏢 Real-World System Choices (Beyond Textbook Examples)

These are publicly documented engineering choices from companies you use every day:

| Company / Product | Database | CAP side | Why |
|-------------------|----------|----------|-----|
| **GitHub** | MySQL (Vitess) | **CP** | Issues, PRs, commits must never be lost or reordered |
| **Stripe** | PostgreSQL | **CP** | Money movement requires linearizable transactions |
| **Discord** | Cassandra → ScyllaDB | **AP** | Trillions of messages; missing one for 10 ms is fine |
| **Netflix** | Cassandra (metadata) + S3 | **AP** | Catalog can be stale; service must never go down |
| **Slack** | MySQL + Vitess (sharded) | **CP** | Message ordering inside a channel must be exact |
| **Shopify** | MySQL | **CP** | Inventory and orders need ACID |
| **Uber** | Schemaless on MySQL | **CP** for trips, **AP** for ETAs | Mixed: trip state is CP, location updates are AP |
| **Amazon DynamoDB** (used by Lambda, Alexa) | DynamoDB | **AP by default** | Configurable per query; defaults to fast/eventual |
| **Amazon S3** (since Dec 2020) | S3 | **CP for reads after writes** | AWS rebuilt the index layer to give *strong* read-after-write — historically S3 was eventual! |
| **Google Docs** | Spanner + Bigtable | **CP** (Spanner) + **AP** (presence) | Document content is CP; "who's online" cursor is AP |

### 🕰️ The S3 story (a great interview anecdote)

For its first **14 years**, S3 was *eventually consistent* — after `PUT object`,
a subsequent `GET` could return the old version or a `404`. In December 2020 AWS
released [strong read-after-write consistency](https://aws.amazon.com/s3/consistency/)
**at no extra cost or latency**. They achieved this by adding a new
strongly-consistent metadata layer in front of the existing AP storage. The
lesson: CAP trade-offs are not always a permanent choice — better hardware,
better protocols (Paxos/Raft), and clever architecture can move the needle.


## 📏 The Consistency Spectrum

Consistency isn't just "strong" or "eventual" — there's a whole spectrum:

```
Strongest ◄─────────────────────────────────────────────► Weakest

┌────────────┐ ┌────────────┐ ┌──────────────────┐ ┌────────────┐
│   Strong   │ │   Causal   │ │ Read-Your-Own-   │ │  Eventual  │
│            │ │            │ │     Writes       │ │            │
│ All reads  │ │ Related    │ │ You see your     │ │ Data will  │
│ see latest │ │ events in  │ │ own updates;     │ │ converge   │
│ write      │ │ correct    │ │ others may not   │ │ eventually │
│            │ │ order      │ │                  │ │            │
│ 🐌 Slowest │ │            │ │                  │ │ ⚡ Fastest  │
└────────────┘ └────────────┘ └──────────────────┘ └────────────┘

Cost:  💰💰💰     💰💰        💰💰                   💰
Speed: 🐌         🚶         🏃                     🚀
```

Let's explore each level with code examples.

In [4]:
# Demonstration: Strong Consistency
# All reads go to the PRIMARY — guaranteed fresh, but slower

print("🔒 Strong Consistency: All reads from PRIMARY")
print("=" * 55)
print()

def strong_consistent_read(user_id):
    """Always read from primary — guaranteed latest data."""
    start = time.time()
    conn = get_primary()
    cursor = conn.cursor()
    cursor.execute("SELECT display_name, profile_views FROM user_profiles WHERE id = %s", (user_id,))
    row = cursor.fetchone()
    conn.close()
    latency = (time.time() - start) * 1000
    return row, latency

# Write, then immediately read from primary
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("UPDATE user_profiles SET display_name = 'Alice (Strong Read)' WHERE id = 1")
conn.close()

row, latency = strong_consistent_read(1)
print(f"   Write: 'Alice (Strong Read)'")
print(f"   Read:  '{row[0]}'")
print(f"   Latency: {latency:.1f} ms")
print(f"   ✅ Always fresh — but every read hits the primary (slower, more load)")

🔒 Strong Consistency: All reads from PRIMARY

   Write: 'Alice (Strong Read)'
   Read:  'Alice (Strong Read)'
   Latency: 13.4 ms
   ✅ Always fresh — but every read hits the primary (slower, more load)


In [5]:
# Demonstration: Eventual Consistency
# Reads go to the replica — fast but may be stale

print("🌐 Eventual Consistency: Reads from REPLICA")
print("=" * 55)
print()

def eventually_consistent_read(user_id):
    """Read from replica — fast but may be stale."""
    start = time.time()
    conn = get_replica()
    cursor = conn.cursor()
    cursor.execute("SELECT display_name, profile_views FROM user_profiles WHERE id = %s", (user_id,))
    row = cursor.fetchone()
    conn.close()
    latency = (time.time() - start) * 1000
    return row, latency

# Write to primary
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("UPDATE user_profiles SET display_name = 'Alice (Eventual Read)' WHERE id = 1")
conn.close()

# Immediately read from replica
row, latency = eventually_consistent_read(1)
print(f"   Write: 'Alice (Eventual Read)'")
print(f"   Read:  '{row[0]}'")
print(f"   Latency: {latency:.1f} ms")

if "Eventual" in row[0]:
    print(f"   ✅ Replica caught up quickly!")
else:
    print(f"   ⏳ Stale data — replica hasn't caught up yet")
    print(f"   This is expected with eventual consistency")

# Wait and try again
time.sleep(1)
row2, _ = eventually_consistent_read(1)
print(f"   Read after 1s: '{row2[0]}'")
print(f"   ✅ Eventually converges — that's why it's called 'eventual' consistency!")

🌐 Eventual Consistency: Reads from REPLICA

   Write: 'Alice (Eventual Read)'
   Read:  'Alice (Eventual Read)'
   Latency: 11.9 ms
   ✅ Replica caught up quickly!


   Read after 1s: 'Alice (Eventual Read)'
   ✅ Eventually converges — that's why it's called 'eventual' consistency!


In [6]:
# Demonstration: Causal Consistency
# Comments must appear AFTER the post they reply to

print("🔗 Causal Consistency: Events in correct order")
print("=" * 55)
print()

r = get_redis()

# Simulate a social media timeline using Redis lists
# Causal consistency ensures: reply always appears AFTER the original post

timeline_key = "timeline:demo"
r.delete(timeline_key)

# These events are causally related: reply depends on the original post
events = [
    {"type": "post",  "user": "Alice", "content": "Just learned about CAP theorem!",       "causal_id": 1},
    {"type": "reply", "user": "Bob",   "content": "@Alice Great! It's super important.",  "causal_id": 2, "reply_to": 1},
    {"type": "reply", "user": "Carol", "content": "@Bob Agreed, especially for interviews.", "causal_id": 3, "reply_to": 2},
]

# With causal consistency, we ensure ordering
for event in events:
    r.rpush(timeline_key, json.dumps(event))

# Read the timeline back
print("   Timeline (causally ordered):")
print()
timeline = r.lrange(timeline_key, 0, -1)
for i, item in enumerate(timeline):
    event = json.loads(item)
    indent = "     " if event["type"] == "reply" else "   "
    icon = "💬" if event["type"] == "post" else "  ↳ 💬"
    print(f"{indent}{icon} {event['user']}: {event['content']}")

print()
print("   ✅ Replies always appear AFTER the post they reference.")
print("   Without causal consistency, Bob's reply might appear before Alice's post!")
print()
print("💡 Causal consistency is used by:")
print("   - Chat apps (messages in order)")
print("   - Social media (comments after posts)")
print("   - Collaborative editing (changes in logical order)")

r.delete(timeline_key)

🔗 Causal Consistency: Events in correct order

   Timeline (causally ordered):

   💬 Alice: Just learned about CAP theorem!
       ↳ 💬 Bob: @Alice Great! It's super important.
       ↳ 💬 Carol: @Bob Agreed, especially for interviews.

   ✅ Replies always appear AFTER the post they reference.
   Without causal consistency, Bob's reply might appear before Alice's post!

💡 Causal consistency is used by:
   - Chat apps (messages in order)
   - Social media (comments after posts)
   - Collaborative editing (changes in logical order)


1

## 🏗️ Mixing CP and AP in Real Systems

Real-world applications rarely use just one approach. Different features have different requirements:

### Example: Ticketmaster
```
┌─────────────────────────────────────────────────────────┐
│                    TICKETMASTER                          │
│                                                         │
│  Browsing Events ──── AP (Availability)                 │
│  │ Stale event description? No problem.                 │
│  │ Use: Read replicas, CDN, Redis cache                 │
│  │                                                      │
│  Booking a Seat ───── CP (Consistency)                  │
│  │ Double-booking? DISASTER.                            │
│  │ Use: Single primary, transactions, locking           │
│  │                                                      │
│  Payment ──────────── CP (Consistency)                  │
│    Charge twice? Lawsuit.                               │
│    Use: Idempotent writes, transaction log              │
└─────────────────────────────────────────────────────────┘
```

In [7]:
# Let's build a mini Ticketmaster to demonstrate mixing CP and AP

r = get_redis()

class TicketSystem:
    """A ticket system that uses AP for browsing and CP for booking."""

    def browse_events_ap(self):
        """AP: browse events from cache (fast, possibly stale)."""
        cached = r.get("events:list")
        if cached:
            return json.loads(cached), "cache"

        # Cache miss — fetch from replica (fast reads)
        conn = get_replica()
        cursor = conn.cursor()
        cursor.execute("SELECT id, name, venue, available_seats FROM events")
        events = [{"id": r[0], "name": r[1], "venue": r[2], "seats": r[3]} for r in cursor.fetchall()]
        conn.close()

        # Cache for 30 seconds (AP: stale data is fine)
        r.set("events:list", json.dumps(events), ex=30)
        return events, "replica"

    def book_seat_cp(self, event_id, seat_number, user_id):
        """CP: book a seat using the primary with row locking."""
        conn = get_primary()
        try:
            cursor = conn.cursor()

            # Lock the event row to prevent race conditions
            cursor.execute("SELECT available_seats FROM events WHERE id = %s FOR UPDATE", (event_id,))
            seats = cursor.fetchone()[0]

            if seats <= 0:
                conn.rollback()
                return False, "SOLD OUT"

            # Try to insert reservation (UNIQUE constraint prevents double-booking)
            cursor.execute(
                "INSERT INTO seat_reservations (event_id, seat_number, user_id) VALUES (%s, %s, %s)",
                (event_id, seat_number, user_id)
            )

            # Decrement available seats
            cursor.execute(
                "UPDATE events SET available_seats = available_seats - 1 WHERE id = %s",
                (event_id,)
            )

            conn.commit()

            # Invalidate the cache so browsing shows updated availability
            r.delete("events:list")

            return True, "CONFIRMED"
        except psycopg2.errors.UniqueViolation:
            conn.rollback()
            return False, "SEAT ALREADY TAKEN"
        except Exception as e:
            conn.rollback()
            return False, str(e)
        finally:
            conn.close()

# Demo
ts = TicketSystem()

print("🎫 Mini Ticketmaster: Mixing CP and AP")
print("=" * 60)
print()

# AP: Browse events (fast, from cache or replica)
events, source = ts.browse_events_ap()
print(f"🌐 AP: Browsing events (source: {source})")
for e in events[:3]:
    print(f"   🎵 {e['name']} at {e['venue']} — {e['seats']} seats left")
print()

# CP: Book seats (consistent, uses primary with locking)
print("🔒 CP: Booking seats (must be consistent!)")
bookings = [
    (1, "A1", 1, "Alice"),
    (1, "A2", 2, "Bob"),
    (1, "A1", 3, "Carol"),  # this should FAIL — A1 already taken!
]

for event_id, seat, user_id, name in bookings:
    success, status = ts.book_seat_cp(event_id, seat, user_id)
    icon = "✅" if success else "❌"
    print(f"   {icon} {name} books seat {seat}: {status}")

print()
print("💡 Notice how the system uses BOTH approaches:")
print("   - Browsing is AP → fast, cache-friendly, stale data is fine")
print("   - Booking is CP → slower but SAFE, no double-bookings")

🎫 Mini Ticketmaster: Mixing CP and AP

🌐 AP: Browsing events (source: replica)
   🎵 Taylor Swift – Eras Tour at Madison Square Garden — 20 seats left
   🎵 NBA Finals Game 7 at Chase Center — 15 seats left
   🎵 Broadway: Hamilton at Richard Rodgers Theatre — 10 seats left

🔒 CP: Booking seats (must be consistent!)
   ✅ Alice books seat A1: CONFIRMED
   ✅ Bob books seat A2: CONFIRMED
   ❌ Carol books seat A1: SEAT ALREADY TAKEN

💡 Notice how the system uses BOTH approaches:
   - Browsing is AP → fast, cache-friendly, stale data is fine
   - Booking is CP → slower but SAFE, no double-bookings


## 🏛️ Two Famous Real-World Designs: Dynamo vs Spanner

The two most influential distributed-database papers chose **opposite** sides
of CAP — and both work great. Knowing this story is gold in interviews.

### 🌐 Amazon Dynamo (2007) — picked **AP**

Amazon's shopping cart could not afford to be down on Black Friday, so the
Dynamo paper (which inspired DynamoDB, Cassandra, Riak, Voldemort) chose
**availability over consistency**:

- **Always writable**, even during a partition — every replica accepts writes.
- Conflicts are detected with **vector clocks** and resolved later (sometimes
  by the application: Amazon merges divergent shopping carts by taking the
  union of items).
- Tunable consistency via the **N/R/W quorum**: pick how many replicas hold
  data (N), must agree on a read (R), and must ack a write (W).
  - `R + W > N` gives strong consistency at the cost of latency.
  - `R = W = 1` is fastest but eventual.
- Famous quote: *"An add to cart should never fail."*

### 🌍 Google Spanner (2012) — picked **CP** (and got *both*, kinda)

Google needed strongly consistent global transactions for AdWords. Spanner
uses **TrueTime** — a clock API backed by GPS receivers and atomic clocks in
every data centre — so every server agrees on time within a few milliseconds.

- Cross-continent transactions with **external (linearizable) consistency**.
- During a partition, Spanner **stops accepting writes** in the partitioned
  region rather than diverge → classic CP.
- Achieves >99.999% availability in practice because Google's network rarely
  partitions — but when it does, consistency wins.
- Open-source equivalents: **CockroachDB**, **YugabyteDB**, **TiDB**.

### 📊 Side by side

| Property | Dynamo (AP) | Spanner (CP) |
|----------|-------------|--------------|
| Conflict handling | Accept divergence, reconcile later | Refuse writes that can't be ordered |
| Primary use case | Shopping cart, session store, IoT | Bank ledger, ad auction, inventory |
| Clock requirements | None (logical / vector clocks) | Atomic clocks + GPS (TrueTime) |
| Write latency | Single-digit ms | 10–100 ms (cross-region commit) |
| What "fails" during a partition | Reconciliation (you may see weird data) | Writes (you see errors) |

💡 **Interview takeaway**: there is no "winner". Dynamo and Spanner solve
*different problems*. Picking the right model is the engineering decision.


## 🧭 Decision Framework for Interviews

When the interviewer asks you to design a system, here's how to decide:

### Step 1: Identify Your Features
List the main features of the system.

### Step 2: Classify Each Feature
For each feature, ask: *"Would stale data be catastrophic?"*

### Step 3: Choose Your Databases
Map features to appropriate database choices.

### Step 4: State It Clearly
Say something like: *"For booking, I'll use PostgreSQL for strong consistency. For the event catalog, I'll use Redis caching with eventual consistency for better read performance."*

In [8]:
# Let's build the decision framework as a runnable tool

interview_scenarios = [
    {
        "system": "Ticketmaster (Event Booking)",
        "features": [
            ("Browse events",          "AP", "Stale event info is fine",      "Redis cache + Postgres replica"),
            ("Book a seat",            "CP", "Double-booking is catastrophic", "Postgres primary + transactions"),
            ("Payment processing",     "CP", "Double-charge is a lawsuit",     "Postgres primary + idempotency"),
            ("View booking history",   "AP", "Slight delay is acceptable",     "Postgres replica"),
        ]
    },
    {
        "system": "Twitter (Social Media)",
        "features": [
            ("View timeline",          "AP", "Missing a tweet for 5s is fine",      "Redis + Cassandra"),
            ("Post a tweet",           "AP", "Eventual delivery is acceptable",     "Kafka + Cassandra"),
            ("View follower count",    "AP", "Off by 1 is fine",                    "Redis counter"),
            ("Send a DM",             "CP", "Messages must not be lost/reordered", "Postgres + Kafka"),
        ]
    },
    {
        "system": "Uber (Ride Sharing)",
        "features": [
            ("Show nearby drivers",    "AP", "Approximate location is fine",   "Redis GEO + eventual sync"),
            ("Match rider to driver",  "CP", "Double-matching is a disaster",  "Postgres + distributed lock"),
            ("Calculate fare",         "CP", "Wrong fare = angry customers",   "Postgres primary"),
            ("Trip history",           "AP", "Slight delay is acceptable",     "Cassandra / DynamoDB"),
        ]
    },
]

print("🧭 CAP Decision Framework: Interview Scenarios")
print("=" * 80)

for scenario in interview_scenarios:
    print(f"\n🏢 {scenario['system']}")
    print(f"   {'Feature':<25} {'CAP':>4}  {'Reasoning':<35} Technology")
    print("   " + "-" * 75)
    for feature, cap, reason, tech in scenario["features"]:
        icon = "🔒" if cap == "CP" else "🌐"
        print(f"   {feature:<25} {icon} {cap:>2}  {reason:<35} {tech}")

print()
print("💡 Key Interview Phrases:")
print('   "For [feature X], I\'ll prioritize consistency because [stale data risk]."')
print('   "For [feature Y], I\'ll prioritize availability since [stale data is harmless]."')

🧭 CAP Decision Framework: Interview Scenarios

🏢 Ticketmaster (Event Booking)
   Feature                    CAP  Reasoning                           Technology
   ---------------------------------------------------------------------------
   Browse events             🌐 AP  Stale event info is fine            Redis cache + Postgres replica
   Book a seat               🔒 CP  Double-booking is catastrophic      Postgres primary + transactions
   Payment processing        🔒 CP  Double-charge is a lawsuit          Postgres primary + idempotency
   View booking history      🌐 AP  Slight delay is acceptable          Postgres replica

🏢 Twitter (Social Media)
   Feature                    CAP  Reasoning                           Technology
   ---------------------------------------------------------------------------
   View timeline             🌐 AP  Missing a tweet for 5s is fine      Redis + Cassandra
   Post a tweet              🌐 AP  Eventual delivery is acceptable     Kafka + Cassandra
 

In [9]:
# Let's compare read performance: primary (CP) vs replica (AP) vs Redis (AP)

def benchmark(label, read_fn, iterations=100):
    """Measure average read latency."""
    times = []
    for _ in range(iterations):
        start = time.time()
        read_fn()
        times.append((time.time() - start) * 1000)
    avg = sum(times) / len(times)
    return avg, min(times), max(times)

# Prepare Redis cache
r = get_redis()
conn = get_primary()
cursor = conn.cursor()
cursor.execute("SELECT id, display_name, bio FROM user_profiles WHERE id = 1")
row = cursor.fetchone()
conn.close()
r.set("user:1:profile", json.dumps({"id": row[0], "name": row[1], "bio": row[2]}))

def read_from_primary():
    conn = get_primary()
    cursor = conn.cursor()
    cursor.execute("SELECT display_name, bio FROM user_profiles WHERE id = 1")
    cursor.fetchone()
    conn.close()

def read_from_replica():
    conn = get_replica()
    cursor = conn.cursor()
    cursor.execute("SELECT display_name, bio FROM user_profiles WHERE id = 1")
    cursor.fetchone()
    conn.close()

def read_from_redis():
    data = r.get("user:1:profile")
    json.loads(data)

print("⚡ Read Performance Comparison (100 reads each)")
print("=" * 55)
print()

for label, fn, cap_type in [
    ("Primary (CP - strong)",    read_from_primary, "CP"),
    ("Replica (AP - eventual)",  read_from_replica, "AP"),
    ("Redis (AP - cached)",      read_from_redis,   "AP"),
]:
    avg, mn, mx = benchmark(label, fn)
    icon = "🔒" if cap_type == "CP" else "🌐"
    print(f"   {icon} {label:<28} avg: {avg:>6.2f} ms  min: {mn:>5.2f} ms  max: {mx:>6.2f} ms")

print()
print("💡 CP (primary) is slower because it's the single source of truth.")
print("   AP (replica/Redis) is faster because it trades freshness for speed.")
print("   This speed difference matters enormously at scale (millions of reads/sec).")

⚡ Read Performance Comparison (100 reads each)



   🔒 Primary (CP - strong)        avg:  12.15 ms  min: 10.26 ms  max:  22.20 ms


   🌐 Replica (AP - eventual)      avg:  13.44 ms  min: 10.76 ms  max:  17.37 ms
   🌐 Redis (AP - cached)          avg:   0.27 ms  min:  0.21 ms  max:   0.53 ms

💡 CP (primary) is slower because it's the single source of truth.
   AP (replica/Redis) is faster because it trades freshness for speed.
   This speed difference matters enormously at scale (millions of reads/sec).


## 🎯 Quick Reference: The One-Minute CAP Cheat Sheet

Use this in interviews when making database decisions:

| Question | If YES → CP | If NO → AP |
|----------|-------------|------------|
| Can stale data cause financial loss? | ✅ Bank, trading, billing | |
| Can stale data cause double-booking? | ✅ Tickets, inventory | |
| Can stale data cause safety issues? | ✅ Medical records, auth | |
| Is the data just informational? | | ✅ Profiles, feeds, reviews |
| Can the user tolerate a 5-second delay? | | ✅ Notifications, analytics |
| Do you need 99.99% uptime? | | ✅ Public APIs, status pages |

### The Golden Rule

> **"Does every read need to return the most recent write?"**
> - **Yes** → CP (Consistency priority)
> - **No** → AP (Availability priority)

## 🧹 Cleanup

In [10]:
# Reset all data we modified
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()

# Reset profiles
cursor.execute("UPDATE user_profiles SET display_name = 'Alice Johnson', bio = 'Software engineer who loves distributed systems' WHERE id = 1")

# Reset events
cursor.execute("UPDATE events SET available_seats = total_seats")
cursor.execute("DELETE FROM seat_reservations")

conn.close()

# Clean Redis
r = get_redis()
for pattern in ["user:*", "events:*", "timeline:*", "post:*"]:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)

print("🧹 Cleaned up all demo data")

🧹 Cleaned up all demo data


## 📚 Summary

### Key Takeaways

1. **Database choice = CAP choice** — PostgreSQL/Spanner are CP, Cassandra/Redis are AP, DynamoDB is configurable
2. **Consistency is a spectrum** — strong, causal, read-your-own-writes, and eventual
3. **Real systems mix CP and AP** — different features have different requirements
4. **Interview framework**: Identify features → classify each as CP or AP → choose databases → state your reasoning
5. **The golden question**: "Does every read need to return the most recent write?"

### What You've Learned in This Series

| Notebook | Key Concept |
|----------|-------------|
| **1 - Understanding CAP** | What C, A, P mean; why P is required; CP vs AP |
| **2 - Consistency vs Availability** | Replication lag, simulated partitions, CP vs AP in code |
| **3 - Choosing the Right Database** | Database CAP mapping, consistency spectrum, interview framework |

### Further Reading

- [Hello Interview – CAP Theorem](https://www.hellointerview.com/learn/system-design/core-concepts/cap-theorem)
- [Hello Interview – PostgreSQL Deep Dive](https://www.hellointerview.com/learn/system-design/deep-dives/postgres)
- [Hello Interview – DynamoDB Deep Dive](https://www.hellointerview.com/learn/system-design/deep-dives/dynamodb)
- [Hello Interview – Redis Deep Dive](https://www.hellointerview.com/learn/system-design/deep-dives/redis)
- [Hello Interview – Cassandra Deep Dive](https://www.hellointerview.com/learn/system-design/deep-dives/cassandra)